# Fake News Detection: Loss Curves and Evaluation Figures (reproducible)

This notebook rebuilds, from scratch, every diagnostic figure in Chapter 4 of the thesis: the **loss curves (Fig. 4.4)**, the confusion matrices, the ROC curves and the precision-recall curves. Nothing here is hard-coded. Each number and curve is produced by training the models and reading the real predictions.

**What it does**
1. Downloads the two datasets from the Hugging Face Hub: ISOT (headlines) and LIAR2 (statements).
2. Cleans them (removes the "(Reuters)" dateline, URLs and duplicate headlines) and makes a stratified 70:15:15 train, validation and test split (seed 0).
3. Trains three models on each dataset for 3 epochs: **BiLSTM**, **BERT**, **BERT + BiLSTM**.
4. Records the training loss and validation loss after every epoch, restores the weights of the epoch with the lowest validation loss, and predicts the test set.
5. Draws the figures from those recorded values.

**How to run:** set the runtime to a GPU (Runtime > Change runtime type > T4 GPU), then choose Runtime > Run all. A full run takes roughly 30 to 40 minutes on a T4. To try it in about two minutes first, set `QUICK = True` in the settings cell.

Small differences from the thesis numbers (in the third decimal) are normal, because the thesis runs used a different GPU back end.

In [ ]:
!pip -q install transformers huggingface_hub scikit-learn matplotlib

In [ ]:
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from sklearn.metrics import (accuracy_score, auc, confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from torch import nn
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

QUICK = False
SEED, EPOCHS, BATCH, MAX_LEN = 0, 3, 32, 64
BERT = "bert-base-uncased"
OUT = Path("results")
OUT.mkdir(exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV)

## 1. Data

ISOT label 0 means fake. For LIAR2 the original labels 0, 1 and 2 (pants-on-fire, false, barely-true) are treated as fake, and 3, 4 and 5 as real. In the arrays below `1` always means fake.

In [ ]:
ISOT_REPO = "ErfanMoosaviMonazzah/fake-news-detection-dataset-English"
LIAR2_REPO = "chengxuphd/liar2"
DATELINE = re.compile(r"^[A-Z][A-Za-z .,/'-]{1,40}\s*\((reuters)\)\s*[-\u2013\u2014]?\s*", re.I)


def clean(text):
    text = DATELINE.sub("", str(text))
    text = re.sub(r"\breuters\b", " ", text, flags=re.I)
    text = re.sub(r"https?://\S+|pic\.twitter\.com/\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def load(name):
    if name == "isot":
        files = [hf_hub_download(ISOT_REPO, f"{s}.tsv", repo_type="dataset") for s in ("train", "validation", "test")]
        df = pd.concat([pd.read_csv(f, sep="\t") for f in files])
        df = df.dropna(subset=["title"]).drop_duplicates(subset="title")
        out = pd.DataFrame({"text": df["title"], "label": (df["label"] == 0).astype(int)})
    else:
        files = [hf_hub_download(LIAR2_REPO, f"{s}.csv", repo_type="dataset") for s in ("train", "valid", "test")]
        df = pd.concat([pd.read_csv(f) for f in files])
        out = pd.DataFrame({"text": df["statement"], "label": df["label"].isin({0, 1, 2}).astype(int)})
    out["text"] = out["text"].apply(clean)
    return out[out["text"].str.len() > 0].reset_index(drop=True)


def split(df, seed):
    tr, rest = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=seed)
    va, te = train_test_split(rest, test_size=0.5, stratify=rest["label"], random_state=seed)
    return tr, va, te


DATA = {}
for name in ("isot", "liar"):
    df = load(name)
    tr, va, te = split(df, SEED)
    if QUICK:
        tr, va, te = tr.sample(600, random_state=0), va.sample(200, random_state=0), te.sample(300, random_state=0)
    DATA[name] = (tr, va, te)
    print(f"{name}: {len(df)} samples, fake share {df['label'].mean():.3f} | train/val/test = {len(tr)}/{len(va)}/{len(te)}")

## 2. Models

* **BiLSTM:** a 2-layer bidirectional LSTM (hidden size 128, dropout 0.5) on randomly initialised word-piece embeddings, with max-pooling over time.
* **BERT:** `bert-base-uncased` fine-tuned with a classification head.
* **BERT + BiLSTM:** the full sequence of BERT's contextual embeddings goes into a BiLSTM (hidden size 128), then a dense layer with ReLU, dropout 0.3 and the output layer.

All three use cross-entropy loss, AdamW, a linear learning-rate schedule with 6% warm-up, gradient clipping at 1.0, batch size 32, maximum length 64 and 3 epochs.

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab, dim=128, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.lstm = nn.LSTM(dim, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.5)
        self.drop = nn.Dropout(0.5)
        self.out = nn.Linear(2 * hidden, 2)

    def forward(self, ids, mask):
        seq, _ = self.lstm(self.emb(ids))
        pooled = seq.masked_fill(~mask.bool().unsqueeze(-1), -1e4).max(dim=1).values
        return self.out(self.drop(pooled))


class BertClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = AutoModelForSequenceClassification.from_pretrained(BERT, num_labels=2)

    def forward(self, ids, mask):
        return self.net(input_ids=ids, attention_mask=mask).logits


class BertBiLSTMClassifier(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.bert = AutoModel.from_pretrained(BERT)
        self.drop_in = nn.Dropout(0.1)
        self.lstm = nn.LSTM(self.bert.config.hidden_size, hidden, batch_first=True, bidirectional=True)
        self.dense = nn.Linear(2 * hidden, hidden)
        self.drop_out = nn.Dropout(0.3)
        self.out = nn.Linear(hidden, 2)

    def forward(self, ids, mask):
        states = self.drop_in(self.bert(input_ids=ids, attention_mask=mask).last_hidden_state)
        seq, _ = self.lstm(states)
        pooled = seq.masked_fill(~mask.bool().unsqueeze(-1), -1e4).max(dim=1).values
        return self.out(self.drop_out(torch.relu(self.dense(pooled))))


SETTINGS = {
    "bilstm": {"lr": 1e-3, "weight_decay": 0.0},
    "bert": {"lr": 2e-5, "weight_decay": 0.01},
    "bert_bilstm": {"lr": 2e-5, "weight_decay": 0.01},
}
BUILD = {"bilstm": lambda tok: BiLSTMClassifier(tok.vocab_size), "bert": lambda tok: BertClassifier(), "bert_bilstm": lambda tok: BertBiLSTMClassifier()}

In [ ]:
def batches(tok, frame, shuffle, rng):
    idx = rng.permutation(len(frame)) if shuffle else np.arange(len(frame))
    texts, labels = frame["text"].tolist(), frame["label"].tolist()
    for i in range(0, len(idx), BATCH):
        j = idx[i:i + BATCH]
        enc = tok([texts[k] for k in j], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        yield enc["input_ids"].to(DEV), enc["attention_mask"].to(DEV), torch.tensor([labels[k] for k in j]).to(DEV)


def evaluate(model, tok, frame, loss_fn):
    model.eval()
    probs, total = [], 0.0
    with torch.no_grad():
        for ids, mask, y in batches(tok, frame, False, None):
            logits = model(ids, mask).float()
            total += loss_fn(logits, y).item() * len(y)
            probs.append(torch.softmax(logits, -1)[:, 1].cpu().numpy())
    return np.concatenate(probs), total / len(frame)


def train_one(name, dataset):
    train, val, test = DATA[dataset]
    torch.manual_seed(SEED)
    rng = np.random.RandomState(SEED)
    tok = AutoTokenizer.from_pretrained(BERT)
    model = BUILD[name](tok).to(DEV)
    cfg = SETTINGS[name]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    steps = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    loss_fn = nn.CrossEntropyLoss()
    train_loss, val_loss, best, best_state = [], [], float("inf"), None
    for epoch in range(EPOCHS):
        model.train()
        t0, running, seen = time.time(), 0.0, 0
        for ids, mask, y in batches(tok, train, True, rng):
            loss = loss_fn(model(ids, mask).float(), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad()
            running += loss.item() * len(y)
            seen += len(y)
        _, vloss = evaluate(model, tok, val, loss_fn)
        train_loss.append(running / seen)
        val_loss.append(vloss)
        print(f"  {dataset} {name} epoch {epoch + 1}: train loss {running / seen:.4f} | validation loss {vloss:.4f} | {time.time() - t0:.0f}s", flush=True)
        if vloss < best:
            best, best_state = vloss, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    probs, test_loss = evaluate(model, tok, test, loss_fn)
    return {"y": test["label"].values, "p": probs, "train_loss": train_loss, "val_loss": val_loss, "test_loss": test_loss}

## 3. Train all six runs (2 datasets x 3 models)

The per-epoch training and validation loss printed here are exactly the values plotted in the loss-curve figure below.

In [ ]:
RESULTS = {}
for dataset in ("isot", "liar"):
    for name in ("bilstm", "bert", "bert_bilstm"):
        print(f"== {dataset} / {name}")
        RESULTS[(dataset, name)] = train_one(name, dataset)

## 5. Plotting functions

The same functions draw the figures twice: once from the models trained above (Section 6) and once from the **saved predictions behind the thesis figures** (Section 8). That is how the notebook shows that the thesis figures come from real data and not from made-up numbers.

In [ ]:
DATASETS = [("isot", "ISOT"), ("liar", "LIAR2")]
MODELS = [("bilstm", "BiLSTM", "#d95f02"), ("bert", "BERT", "#1b9e77"), ("bert_bilstm", "BERT+BiLSTM", "#7570b3")]


def plot_loss(R, fname):
    fig, axes = plt.subplots(2, 3, figsize=(12, 7.5))
    for r, (ds, ds_label) in enumerate(DATASETS):
        for c, (m, m_label, _) in enumerate(MODELS):
            d = R[(ds, m)]
            epochs = np.arange(1, len(d["train_loss"]) + 1)
            best = int(np.argmin(d["val_loss"]))
            ax = axes[r][c]
            ax.plot(epochs, d["train_loss"], marker="o", color="#1f77b4", label="Training loss")
            ax.plot(epochs, d["val_loss"], marker="s", color="#ff7f0e", label="Validation loss")
            ax.scatter(epochs[best], d["val_loss"][best], marker="*", s=220, color="#d62728", zorder=5, label="Best validation epoch")
            for e, t, v in zip(epochs, d["train_loss"], d["val_loss"]):
                above = t >= v
                ax.annotate(f"{t:.3f}", (e, t), textcoords="offset points", xytext=(0, 9 if above else -14), ha="center", fontsize=7, color="#1f77b4")
                ax.annotate(f"{v:.3f}", (e, v), textcoords="offset points", xytext=(0, -14 if above else 9), ha="center", fontsize=7, color="#ff7f0e")
            ax.margins(y=0.2)
            ax.set_xticks(epochs)
            ax.set_title(f"{ds_label}: {m_label}", fontsize=11)
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Cross-entropy loss")
            ax.grid(alpha=0.3)
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=10, frameon=False)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(OUT / fname, dpi=200, bbox_inches="tight")
    plt.show()


def plot_confusion(R, fname):
    fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
    for r, (ds, ds_label) in enumerate(DATASETS):
        for c, (m, m_label, _) in enumerate(MODELS):
            d = R[(ds, m)]
            cm = confusion_matrix(d["y"], (d["p"] >= 0.5).astype(int), labels=[0, 1])
            ax = axes[r][c]
            ax.imshow(cm, cmap="Blues")
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
            ax.set_xticks([0, 1], ["Real", "Fake"])
            ax.set_yticks([0, 1], ["Real", "Fake"])
            ax.set_title(f"{ds_label}: {m_label}", fontsize=10)
            ax.set_xlabel("Predicted")
            ax.set_ylabel("Actual")
    fig.tight_layout()
    fig.savefig(OUT / fname, dpi=200, bbox_inches="tight")
    plt.show()


def plot_curves(R, kind, fname):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, (ds, ds_label) in zip(axes, DATASETS):
        for m, m_label, colour in MODELS:
            d = R[(ds, m)]
            if kind == "roc":
                x, y, _ = roc_curve(d["y"], d["p"])
                ax.plot(x, y, color=colour, label=f"{m_label} (AUC {auc(x, y):.3f})")
            else:
                prec, rec, _ = precision_recall_curve(d["y"], d["p"])
                ax.plot(rec, prec, color=colour, label=f"{m_label} (AUC {auc(rec, prec):.3f})")
        if kind == "roc":
            ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
            ax.set_xlabel("False positive rate")
            ax.set_ylabel("True positive rate")
        else:
            ax.set_xlabel("Recall")
            ax.set_ylabel("Precision")
        ax.set_title(ds_label)
        ax.legend(loc="lower right" if kind == "roc" else "lower left", fontsize=8)
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / fname, dpi=200, bbox_inches="tight")
    plt.show()


def metrics_table(R):
    rows = []
    for (dataset, name), r in R.items():
        pred = (r["p"] >= 0.5).astype(int)
        rows.append({"dataset": "ISOT" if dataset == "isot" else "LIAR2", "model": name,
                     "accuracy": accuracy_score(r["y"], pred), "precision": precision_score(r["y"], pred, zero_division=0),
                     "recall": recall_score(r["y"], pred, zero_division=0), "f1": f1_score(r["y"], pred, zero_division=0),
                     "auc": roc_auc_score(r["y"], r["p"]), "test_loss": r["test_loss"]})
    return pd.DataFrame(rows).round(4)

## 6. Results of the models trained in this notebook

### 6.1 Test-set metrics

In [ ]:
metrics = metrics_table(RESULTS)
metrics.to_csv(OUT / "metrics.csv", index=False)
metrics

### 6.2 Loss curves (Fig. 4.4)

Blue is the training loss and orange the validation loss after each epoch. The red star marks the epoch with the lowest validation loss, whose weights are used for testing. **Every number printed on the curves is a value measured during training above. Nothing is typed in by hand.**

**How to read it**
* **ISOT:** training loss falls steadily while validation loss stays low and almost flat, so the models fit the data well. After the best epoch the validation loss stops improving or rises slightly while the training loss keeps falling, which is the start of overfitting and the reason the best epoch is restored.
* **LIAR2:** training loss keeps falling but validation loss does **not**. For all three models it is lowest at epoch 1 and then rises, so the models start to memorise the training statements after one epoch instead of learning patterns that carry over. This matches the lower LIAR2 accuracy of about 0.66 to 0.69.

In [ ]:
plot_loss(RESULTS, "Loss.png")
table = pd.DataFrame([{"dataset": ds_label, "model": m_label, "epoch": e + 1,
                       "train_loss": RESULTS[(ds, m)]["train_loss"][e], "val_loss": RESULTS[(ds, m)]["val_loss"][e]}
                      for ds, ds_label in DATASETS for m, m_label, _ in MODELS for e in range(EPOCHS)]).round(4)
table.to_csv(OUT / "loss_values.csv", index=False)
table

### 6.3 Confusion matrices (Fig. 4.1), ROC (Fig. 4.2) and precision-recall (Fig. 4.3)

The confusion matrices count the test samples in each cell, and the ROC and PR curves are computed from the predicted probability of every test sample, `sklearn.metrics.roc_curve` and `precision_recall_curve`.

In [ ]:
plot_confusion(RESULTS, "Confusion.png")
plot_curves(RESULTS, "roc", "ROC.png")
plot_curves(RESULTS, "pr", "PR_Curve.png")

## 7. Where does every figure in the report come from?

| Figure in the report | Data source | Produced by | Real data? |
|---|---|---|---|
| Fig. 4.1 Confusion matrices | test-set predictions of the 6 runs | this notebook (Section 6.3), thesis copy from `make_figures.py` | Yes |
| Fig. 4.2 ROC curves | probabilities of the same test predictions | this notebook (Section 6.3) | Yes |
| Fig. 4.3 PR curves | probabilities of the same test predictions | this notebook (Section 6.3) | Yes |
| Fig. 4.4 Loss curves | training and validation loss measured per epoch | this notebook (Section 6.2) | Yes |
| GRU bar chart (accuracy, precision, recall, F1) | `experiments/results/gru*.json`, written by the five GRU training notebooks | Section 9 | Yes, saved test metrics |
| Cross-dataset table | `experiments/results/cross_dataset/*.csv`, written by `finetune_transformer.py` (3 seeds) | Section 10 | Yes |
| Architecture drawings (Arch.jpg, architecture.png) | diagrams of the model, not data | drawn by hand | Not data |

Earlier drafts of the report used plots built from artificial predictions. They were replaced by the figures above, which are generated from real model outputs. Section 8 proves it by redrawing the thesis figures from the saved predictions.

## 8. Proof that the thesis figures come from real saved predictions

The thesis figures were made on the author's PC (AMD GPU) by `train_three_models.py` and `make_figures.py`. That script stored every test label, every predicted probability and every epoch's loss in `experiments/results/figures_data/*.npz` in the GitHub repository. The next cells download those files, redraw the figures from them with the same functions, and recompute the table numbers printed in the thesis.

In [ ]:
import io
import urllib.request

RAW = "https://raw.githubusercontent.com/mdakashhossain1/m.tech/main/"


def fetch(path):
    with urllib.request.urlopen(RAW + path) as resp:
        return resp.read()


THESIS = {}
for dataset in ("isot", "liar"):
    for name in ("bilstm", "bert", "bert_bilstm"):
        z = np.load(io.BytesIO(fetch(f"experiments/results/figures_data/{dataset}_{name}.npz")))
        THESIS[(dataset, name)] = {k: z[k] for k in z.files}
        THESIS[(dataset, name)]["test_loss"] = float(z["test_loss"])
print("loaded", len(THESIS), "saved runs; test samples per run:", {k[0] + "/" + k[1]: len(v["y"]) for k, v in THESIS.items()})

In [ ]:
REPORTED = {  # accuracy and F1 printed in Tables 4.3 and 4.4 of the thesis
    ("ISOT", "bilstm"): (0.9714, 0.9682), ("ISOT", "bert"): (0.9768, 0.9742), ("ISOT", "bert_bilstm"): (0.9815, 0.9795),
    ("LIAR2", "bilstm"): (0.664, 0.726), ("LIAR2", "bert"): (0.689, 0.713), ("LIAR2", "bert_bilstm"): (0.682, 0.707),
}
saved = metrics_table(THESIS)
saved["thesis_accuracy"] = [REPORTED[(r.dataset, r.model)][0] for r in saved.itertuples()]
saved["thesis_f1"] = [REPORTED[(r.dataset, r.model)][1] for r in saved.itertuples()]
saved["matches_table"] = [(abs(r.accuracy - r.thesis_accuracy) < 0.0006) and (abs(r.f1 - r.thesis_f1) < 0.0006) for r in saved.itertuples()]
saved[["dataset", "model", "accuracy", "thesis_accuracy", "f1", "thesis_f1", "matches_table"]]

In [ ]:
plot_loss(THESIS, "Loss_from_saved_thesis_run.png")
plot_confusion(THESIS, "Confusion_from_saved_thesis_run.png")

Read the loss curves above next to Section 6.2. The shapes agree (training loss falls, and on LIAR2 the validation loss is lowest at epoch 1 and then rises). Small differences in the third decimal come from the different GPU back end and random number streams, not from a different method.

## 9. GRU comparison chart

The GRU models were trained in Keras with the notebooks `gru.ipynb`, `gru_lstm.ipynb`, `gru_cnn.ipynb`, `gru_bilstm.ipynb` and `gru_resnet.ipynb` in the repository (headline only, 10,000-word vocabulary, 40 tokens, 12 epochs at most with early stopping). Each notebook saved its test metrics and its full training history in a JSON file. The cell below reads those files and draws the bar chart used in the report.

In [ ]:
import json

GRU_FILES = ["gru", "gru_lstm", "gru_cnn", "gru_bilstm", "gru_resnet"]
gru_rows, gru_hist = [], {}
for f in GRU_FILES:
    d = json.loads(fetch(f"experiments/results/{f}.json"))
    gru_rows.append({"model": d["model"], **d["metrics"]})
    gru_hist[d["model"]] = d["history"]
gru_df = pd.DataFrame(gru_rows).sort_values("f1_score", ascending=False).reset_index(drop=True)
display(gru_df.round(4))

fig, ax = plt.subplots(figsize=(12, 6))
width = 0.2
for i, metric in enumerate(["accuracy", "precision", "recall", "f1_score"]):
    ax.bar([p + i * width for p in range(len(gru_df))], gru_df[metric], width=width, label=metric)
ax.set_xticks([p + width * 1.5 for p in range(len(gru_df))])
ax.set_xticklabels(gru_df["model"], rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("Model comparison: Accuracy / Precision / Recall / F1")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, h in gru_hist.items():
    ax.plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], marker="o", markersize=3, label=f"{name} (validation loss)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss")
ax.set_title("Validation loss per epoch, recorded during GRU training")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

## 10. Cross-dataset table

These files come from `finetune_transformer.py`, which fine-tunes BERT with 3 seeds. The GRU rows come from the earlier GRU cross-dataset run (`summary.csv`).

In [ ]:
frames = []
for part in ("isot", "liar", "mixed"):
    df = pd.read_csv(io.BytesIO(fetch(f"experiments/results/cross_dataset/bert-base-uncased_{part}.csv")))
    frames.append(df)
bert_runs = pd.concat(frames)
print("BERT runs:", len(bert_runs), "rows over seeds", sorted(bert_runs["seed"].unique()))
bert_runs.groupby(["train", "test"])[["accuracy", "f1", "auc"]].agg(["mean", "std"]).round(3)

In [ ]:
pd.read_csv(io.BytesIO(fetch("experiments/results/cross_dataset/summary.csv")), header=[0, 1], index_col=[0, 1]).round(3)

## 11. Download everything

This packs the figures and the CSV files into one zip.

In [ ]:
import shutil
shutil.make_archive("fake_news_results", "zip", OUT)
try:
    from google.colab import files
    files.download("fake_news_results.zip")
except ImportError:
    print("Saved fake_news_results.zip")